In [1]:
import h5py
import torch
import numpy as np
import os
import json

import keyrank_rs

import torch.nn as nn

from tqdm import tqdm

In [2]:
if os.name == "nt":
    DATAFOLDER = "C:/Data"
else:
    DATAFOLDER = "/mnt/c/Data"

val_test_hdf = h5py.File(f"{DATAFOLDER}/simpleserial-aes-500-diff.hdf5")

val_test_traces = torch.Tensor(np.array(val_test_hdf['trace']))
val_test_plaintexts = torch.Tensor(np.array(val_test_hdf['data']))
val_test_keys = torch.Tensor(np.array(val_test_hdf['key']))

device = torch.device("cuda")

In [3]:
print(val_test_traces.shape)
print(val_test_plaintexts.shape)
print(val_test_keys.shape)

torch.Size([1000, 500, 5000])
torch.Size([1000, 500, 16])
torch.Size([1000, 16])


In [4]:
def metadata_best_epoch(model_name) -> int:
    with open(f"models/{model_name}/metadata.json") as f:
        metadata = json.load(f)
        val_scores = metadata["scores"][1]
        best_epoch = np.array(val_scores).argmin()
    return best_epoch.item()

def get_traces_mean_std(trace_start, trace_end):
    """Load the mean and std of the training trace set within the given interval"""
    with open(f"misc/standardization_tinyaes/trace{trace_start}_{trace_end}.json") as f:
        info = json.load(f)
        mean = info['training_traces_mean']
        std = info['training_traces_std']

    return mean, std

In [5]:
IMPL = "tinyaes"
ARCH = "zhang"
PREDICTION_TARGET = "sbox"
TARGET_BYTE_IDX = 1
TRACE_START = 1000
TRACE_END = 2000
SEED = 777

model_name = f"{IMPL}-{PREDICTION_TARGET}-byte{TARGET_BYTE_IDX}-{ARCH}-{TRACE_START}_{TRACE_END}-s{SEED}"

epoch = metadata_best_epoch(model_name)

model_path = f"models/{model_name}/epoch{epoch}.pt"
print(model_path)

model = torch.load(model_path).to()

models/tinyaes-sbox-byte1-zhang-1000_2000-s777/epoch48.pt


C:\Users\Ulrik\AppData\Local\Temp\ipykernel_16980\2397910419.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load(model_path).to()


In [6]:
sample = 259

traces_mean, traces_std = get_traces_mean_std(TRACE_START, TRACE_END)

traces = (val_test_traces[sample, :, TRACE_START:TRACE_END] - traces_mean) / traces_std
plaintexts = val_test_plaintexts[sample, :, :16] # first plaintext block
key = val_test_keys[sample]

print(traces.shape)
print(plaintexts.shape)
print(key.shape)

torch.Size([500, 1000])
torch.Size([500, 16])
torch.Size([16])


In [7]:
"""Compute traces needed for 99% accurate full key recovery using single model"""

log_softmax = nn.LogSoftmax(dim=1)

# Sample_idx, n_traces
success_matrix = torch.zeros(500,500)


for sample_idx in tqdm(range(0,500)):

    traces_ = (val_test_traces[sample_idx, :, TRACE_START:TRACE_END] - traces_mean) / traces_std
    plaintexts_B1_ = val_test_plaintexts[sample_idx, :, :16] # first plaintext block
    true_key_ = val_test_keys[sample_idx].long()

    sbox_scores = model(traces_.to(device))
    numpy_sbox_scores = sbox_scores.detach().cpu().numpy()

    full_guesses = torch.zeros(500,16)

    for subkey in range(16):
        plaintext_B1_bytes = plaintexts_B1_[:, subkey]
        plaintext_B1_bytes = plaintext_B1_bytes.long().detach().cpu().numpy().squeeze()

        numpy_keyscores1 = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_B1_bytes, numpy_sbox_scores)

        x = torch.Tensor(numpy_keyscores1)
        x = log_softmax(x)
        x = x.cumsum(dim=0)
        guesses = x.argmax(dim=1)
        full_guesses[:, subkey] = guesses

    
    successes = (full_guesses.long() == true_key_).all(dim=1)
    success_matrix[sample_idx, :] = successes


(success_matrix.sum(dim=0) >= 495).nonzero()[0].item()

  0%|          | 0/500 [00:00<?, ?it/s]

 27%|██▋       | 137/500 [00:08<00:21, 16.82it/s]


KeyboardInterrupt: 

In [8]:
"""Compute traces needed for 99% accuracy on individual subkeys using single model"""


log_softmax = nn.LogSoftmax(dim=1)

# subkey, sample_idx, n_traces
success_matrix = torch.zeros(16,500,500)

for sample_idx in tqdm(range(0,500)):

    traces_ = (val_test_traces[sample_idx, :, TRACE_START:TRACE_END] - traces_mean) / traces_std
    plaintexts_B1_ = val_test_plaintexts[sample_idx, :, :16] # first plaintext block
    plaintexts_B2_ = val_test_plaintexts[sample_idx, :, 16:] # second plaintext block
    true_key_ = val_test_keys[sample_idx].long()

    sbox_scores = model(traces_.to(device))
    numpy_sbox_scores = sbox_scores.detach().cpu().numpy()

    for subkey in range(16):
        plaintext_B1_bytes = plaintexts_B1_[:, subkey]
        plaintext_B1_bytes = plaintext_B1_bytes.long().detach().cpu().numpy().squeeze()

        numpy_keyscores1 = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_B1_bytes, numpy_sbox_scores)

        x = torch.Tensor(numpy_keyscores1)
        x = log_softmax(x)
        x = x.cumsum(dim=0)
        guesses = x.argmax(dim=1)

        success = (guesses.long() == true_key_[subkey])
        success_matrix[subkey, sample_idx] = success


n_traces_needed = []

for subkey in range(16):
    nz = (success_matrix[subkey].sum(dim=0) >= 495).nonzero()
    n_traces = nz[0].item() if nz.numel() > 0 else "X"
    n_traces_needed.append(n_traces)

n_traces_needed

100%|██████████| 500/500 [00:28<00:00, 17.60it/s]


['X', 3, 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X']

In [10]:


print("Subkey: &",  " & ".join([f"${n}$" for n in range(16)]), r"\\")
#print("\\hline")
#print("Mean traces: &", " & ".join([f"${mean:.01f}$" for mean in mean_traces_needed]), r"\\")
print("\\hline")
print("Traces 99\\%: &", " & ".join([f"${n_traces}$" for n_traces in n_traces_needed]), r"\\")


#for idx, (mean, n99acc) in enumerate(zip(mean_traces_needed, traces_needed_99acc)):
#    print(f"Subkey {idx:02}, mean: {mean:.03f}, traces needed for 99%: {n99acc}")

Subkey: & $0$ & $1$ & $2$ & $3$ & $4$ & $5$ & $6$ & $7$ & $8$ & $9$ & $10$ & $11$ & $12$ & $13$ & $14$ & $15$ \\
\hline
Traces 99\%: & $X$ & $3$ & $X$ & $X$ & $X$ & $X$ & $X$ & $X$ & $X$ & $X$ & $X$ & $X$ & $X$ & $X$ & $X$ & $X$ \\


In [13]:
print("True key:", key.long().tolist())
true_key =  key.long().tolist()

sbox_scores = model(traces.to(device))
numpy_scores_ = sbox_scores.detach().cpu().numpy()

log_softmax = nn.LogSoftmax(dim=1)

for n_traces in range(2,500):
    guesses = []

    numpy_sbox_scores = numpy_scores_[:n_traces]

    for idx in range(16):

        plaintext_bytes = plaintexts[:n_traces, idx]
        plaintext_bytes = plaintext_bytes.long().detach().cpu().numpy().squeeze()

        numpy_keyscores = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_bytes, numpy_sbox_scores)

        x = torch.Tensor(numpy_keyscores)
        x = log_softmax(x)
        x = x.sum(dim=0)
        x = x.argmax(dim=0)

        guesses.append(x.item())

    if true_key == guesses:
        n_traces = n_traces
        print(n_traces,"traces")
        break

print("Full attack:",guesses)    

True key: [50, 71, 86, 75, 101, 143, 10, 146, 131, 46, 30, 192, 102, 71, 24, 70]
Full attack: [128, 71, 185, 43, 123, 70, 182, 121, 60, 100, 147, 66, 67, 232, 232, 202]


In [30]:
true_key =  key.long().tolist()

n_traces = 75

guesses = []

sbox_scores = model(traces[:n_traces].to(device))
numpy_scores = sbox_scores.detach().cpu().numpy()

for idx in range(16):

    plaintext_bytes = plaintexts[:n_traces, idx]
    plaintext_bytes = plaintext_bytes.long().detach().cpu().numpy().squeeze()

    numpy_keyscores = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_bytes, numpy_scores)

    x = torch.Tensor(numpy_keyscores)
    x = log_softmax(x)
    x = x.sum(dim=0)
    x = x.argmax(dim=0)

    guesses.append(x.item())

print("True key:", true_key)
print("Full attack:",guesses)

True key: [50, 71, 86, 75, 101, 143, 10, 146, 131, 46, 30, 192, 102, 71, 24, 70]
Full attack: [173, 71, 152, 187, 119, 154, 122, 143, 216, 83, 111, 108, 231, 62, 99, 12]


In [ ]:
n_traces = 42

true_key =  key.long().tolist()
guesses = []

attack_data_folder = f"misc/attack_data/key{sample}"
os.makedirs(os.path.dirname(attack_data_folder), exist_ok=True)

np.save(f"{attack_data_folder}/sbox_scores.npy", numpy_sbox_scores)
np.save(f"{attack_data_folder}/plaintexts.npy", plaintexts[:n_traces])

for idx in range(16):

    plaintext_bytes = plaintexts[:n_traces, idx]
    plaintext_bytes = plaintext_bytes.long().detach().cpu().numpy().squeeze()

    numpy_keyscores = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_bytes, numpy_sbox_scores)

    np.save(f"{attack_data_folder}/keybyte{idx}_scores.npy", numpy_keyscores)

    x = torch.Tensor(numpy_keyscores).softmax(dim=1)
    x = x.log()
    x = x.sum(dim=0)
    x = x.argmax(dim=0)

    guesses.append(x.item())



print("True key:", true_key)
print("Full attack:",guesses)


attack_info = {
    "implementation" : IMPL,
    "architecture" : ARCH,
    "target_variable" : PREDICTION_TARGET,
    "training_target_byte" : TARGET_BYTE_IDX,
    "trace_interval_start" : TRACE_START,
    "trace_interval_end" : TRACE_END,
    "testing_set_index" : sample,
    "attack_traces" : n_traces,
    "true_key" : true_key,
    "attack_output" : guesses,
}

with open(f"{attack_data_folder}/attack_info.json", 'w') as f:
    json.dump(attack_info, f, indent=4)

True key: [66, 59, 237, 182, 158, 214, 87, 231, 242, 124, 155, 22, 62, 118, 10, 48]
Full attack: [66, 59, 237, 182, 158, 214, 87, 231, 242, 124, 155, 22, 62, 118, 10, 48]
